# SE-801 Lecture 6 Backpropagation, Initialization, and Stable Training

**Central question:** Why do mathematically valid neural networks fail to train, and how do computational graphs, principled initialization, and dropout enable stable deep learning?

This self-contained notebook accompanies `Lecture_06_Board_and_Speaking_Notes.md`. Predict and calculate before revealing each output. It uses synthetic diagnostic benchmarks and runs offline on CPU with PyTorch, NumPy, and Matplotlib. It embeds every implementation cell and does not import external helper packages.

**Class route:** 80 teaching minutes, 20-minute break, 70 teaching minutes, then 10 minutes of flexibility. RUN 22 is reserve material. Test labels first enter performance reporting at RUN 21.

**Notation:** Batches use row-wise examples: $X \in \mathbb{R}^{N \times d_{\text{in}}}$, weights $W^{(1)} \in \mathbb{R}^{d_{\text{in}} \times d_h}$. The scratch matrix backprop equations mirror PyTorch's internal autograd operations.

**Output folder:** Default `outputs` under the lecture directory.


## RUN 0 Environment setup and reproducible seed

**ASK BEFORE RUNNING:** How does PyTorch represent computational graphs, and how do we ensure reproducible runs?

In [ ]:
# Ensure reproducible output directory and deterministic execution
from pathlib import Path
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(os.environ.get('ANN_L6_OUTPUT_DIR', 'outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Environment ready. PyTorch version: {torch.__version__}, Output directory: {OUTPUT_DIR}")

## RUN 1 Forward computational graph and intermediate caching

**ASK BEFORE RUNNING:** Trace the scalar graph forward. Which intermediate operations must be cached in memory for the backward pass?

In [ ]:
# Computational graph for a scalar composition: L = ln(1 + exp(-(w*x + b)))
# Forward pass caches preactivations for reverse-mode autograd.
x_val = torch.tensor(1.5, requires_grad=False)
w_val = torch.tensor(0.8, requires_grad=True)
b_val = torch.tensor(-0.4, requires_grad=True)

# Forward pass step-by-step
z = w_val * x_val + b_val          # Affine combination (cached)
neg_z = -z                         # Negation
exp_neg_z = torch.exp(neg_z)       # Exponentiation (cached)
one_plus_exp = 1.0 + exp_neg_z     # Denominator term
loss = torch.log(one_plus_exp)     # Binary logistic loss

print(f"Forward pass:")
print(f"  z = w*x + b = {z.item():.4f}")
print(f"  Loss L = ln(1 + exp(-z)) = {loss.item():.4f}")

## RUN 2 Reverse-mode automatic differentiation (Chain rule)

**ASK BEFORE RUNNING:** Calculate dL/dz, dL/dw, and dL/db using the chain rule on paper. Does your result match PyTorch Autograd?

In [ ]:
# Backward manual pass vs PyTorch autograd
# dL/d(one_plus_exp) = 1 / one_plus_exp
# d(one_plus_exp)/d(exp_neg_z) = 1
# d(exp_neg_z)/d(neg_z) = exp_neg_z
# d(neg_z)/dz = -1
# dz/dw = x, dz/db = 1
dL_dloss = 1.0
dL_done_plus_exp = dL_dloss * (1.0 / one_plus_exp.item())
dL_dexp_neg_z = dL_done_plus_exp * 1.0
dL_dneg_z = dL_dexp_neg_z * exp_neg_z.item()
dL_dz = dL_dneg_z * (-1.0)
dL_dw_manual = dL_dz * x_val.item()
dL_db_manual = dL_dz * 1.0

# Autograd verification
loss.backward()
print(f"Manual chain rule:")
print(f"  dL/dz = {dL_dz:.6f}")
print(f"  dL/dw = {dL_dw_manual:.6f}")
print(f"  dL/db = {dL_db_manual:.6f}")
print(f"Autograd results:")
print(f"  w.grad = {w_val.grad.item():.6f}")
print(f"  b.grad = {b_val.grad.item():.6f}")
assert math.isclose(dL_dw_manual, w_val.grad.item(), abs_tol=1e-6)
assert math.isclose(dL_db_manual, b_val.grad.item(), abs_tol=1e-6)
print("Manual gradient matches PyTorch Autograd perfectly!")

## RUN 3 Vectorized 2-layer MLP backpropagation mathematics

**ASK BEFORE RUNNING:** State the exact tensor dimensions for X, W1, b1, Z1, H1, W2, b2, and Z2. How does batch size N propagate?

In [ ]:
# Derivation of shapes and batch matrix equations:
# X: [N, d_in], W1: [d_in, d_h], b1: [d_h]
# Z1 = X W1 + b1,  H1 = phi(Z1)
# W2: [d_h, d_out], b2: [d_out]
# Z2 = H1 W2 + b2,  P = softmax(Z2)
# Loss = Mean Cross-Entropy over N samples
N, d_in, d_h, d_out = 4, 3, 4, 2
torch.manual_seed(101)
X_toy = torch.randn(N, d_in, dtype=torch.float64)
y_toy = torch.tensor([0, 1, 1, 0], dtype=torch.long)

# Weight parameters in double precision for exact tolerance check
W1_scratch = torch.randn(d_in, d_h, dtype=torch.float64, requires_grad=True)
b1_scratch = torch.zeros(d_h, dtype=torch.float64, requires_grad=True)
W2_scratch = torch.randn(d_h, d_out, dtype=torch.float64, requires_grad=True)
b2_scratch = torch.zeros(d_out, dtype=torch.float64, requires_grad=True)

print(f"Shapes:")
print(f"  Input X: {list(X_toy.shape)}, Labels y: {list(y_toy.shape)}")
print(f"  Hidden W1: {list(W1_scratch.shape)}, b1: {list(b1_scratch.shape)}")
print(f"  Output W2: {list(W2_scratch.shape)}, b2: {list(b2_scratch.shape)}")

## RUN 4 Scratch forward pass and softmax cross-entropy

**ASK BEFORE RUNNING:** Compute the forward activations and softmax cross-entropy loss. Why do we subtract max(Z) before exponentiation?

In [ ]:
def scratch_forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1                     # [N, d_h]
    H1 = torch.relu(Z1)                  # [N, d_h]
    Z2 = H1 @ W2 + b2                    # [N, d_out]
    # Numerically stable log-softmax
    max_z = torch.max(Z2, dim=1, keepdim=True)[0]
    exp_z = torch.exp(Z2 - max_z)
    probs = exp_z / torch.sum(exp_z, dim=1, keepdim=True)
    return Z1, H1, Z2, probs

Z1_val, H1_val, Z2_val, probs_val = scratch_forward(X_toy, W1_scratch, b1_scratch, W2_scratch, b2_scratch)
loss_manual = -torch.mean(torch.log(probs_val[torch.arange(N), y_toy]))
print(f"Manual forward pass loss: {loss_manual.item():.6f}")

## RUN 5 Scratch backward pass (Matrix chain rule implementation)

**ASK BEFORE RUNNING:** Derive dL/dZ2, dL/dW2, and backpropagate to dL/dZ1 using the ReLU derivative mask.

In [ ]:
def scratch_backward(X, y, Z1, H1, Z2, probs, W2):
    N = X.shape[0]
    # Gradient of Cross-Entropy with Softmax: dL/dZ2 = (probs - Y_onehot) / N
    dZ2 = probs.clone()
    dZ2[torch.arange(N), y] -= 1.0
    dZ2 = dZ2 / N                        # [N, d_out]
    
    # Gradients for Layer 2:
    # dL/dW2 = H1^T @ dZ2               [d_h, d_out]
    # dL/db2 = sum(dZ2, axis=0)          [d_out]
    dW2 = H1.T @ dZ2
    db2 = torch.sum(dZ2, dim=0)
    
    # Backpropagate to hidden layer:
    # dL/dH1 = dZ2 @ W2^T               [N, d_h]
    # dL/dZ1 = dL/dH1 * phi'(Z1) = dL/dH1 * (Z1 > 0)
    dH1 = dZ2 @ W2.T
    dZ1 = dH1 * (Z1 > 0.0).double()
    
    # Gradients for Layer 1:
    # dL/dW1 = X^T @ dZ1                [d_in, d_h]
    # dL/db1 = sum(dZ1, axis=0)          [d_h]
    dW1 = X.T @ dZ1
    db1 = torch.sum(dZ1, dim=0)
    
    return dW1, db1, dW2, db2

dW1_manual, db1_manual, dW2_manual, db2_manual = scratch_backward(
    X_toy, y_toy, Z1_val, H1_val, Z2_val, probs_val, W2_scratch
)
print("Manual backward pass completed.")
print(f"  Norm(dW1): {torch.norm(dW1_manual).item():.6f}, Norm(dW2): {torch.norm(dW2_manual).item():.6f}")

## RUN 6 Verification of scratch backward pass against PyTorch Autograd

**ASK BEFORE RUNNING:** Compare scratch matrix gradients against autograd. Why is the discrepancy within 1e-15?

In [ ]:
# Compute autograd on the exact same graph
loss_manual.backward()

max_diff_W1 = torch.max(torch.abs(dW1_manual - W1_scratch.grad)).item()
max_diff_b1 = torch.max(torch.abs(db1_manual - b1_scratch.grad)).item()
max_diff_W2 = torch.max(torch.abs(dW2_manual - W2_scratch.grad)).item()
max_diff_b2 = torch.max(torch.abs(db2_manual - b2_scratch.grad)).item()

print(f"Max absolute discrepancy between Scratch and Autograd:")
print(f"  W1: {max_diff_W1:.2e}")
print(f"  b1: {max_diff_b1:.2e}")
print(f"  W2: {max_diff_W2:.2e}")
print(f"  b2: {max_diff_b2:.2e}")
assert max_diff_W1 < 1e-12 and max_diff_b1 < 1e-12
assert max_diff_W2 < 1e-12 and max_diff_b2 < 1e-12
print("Mathematical backward derivation verified to machine precision!")

## RUN 7 Activation saturation & vanishing gradient: Sigmoid vs Tanh vs ReLU

**ASK BEFORE RUNNING:** What is the maximum derivative of Sigmoid vs Tanh vs ReLU? Where does Sigmoid saturate?

In [ ]:
z_range = torch.linspace(-6.0, 6.0, 300, dtype=torch.float32, requires_grad=True)

# Activations and their analytical derivatives
sig = torch.sigmoid(z_range)
tanh = torch.tanh(z_range)
relu = torch.relu(z_range)

# Gradients via autograd
sig_sum = sig.sum()
tanh_sum = tanh.sum()
relu_sum = relu.sum()

sig_grad = torch.autograd.grad(sig_sum, z_range, retain_graph=True)[0]
tanh_grad = torch.autograd.grad(tanh_sum, z_range, retain_graph=True)[0]
relu_grad = torch.autograd.grad(relu_sum, z_range)[0]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(z_range.detach().numpy(), sig.detach().numpy(), label="Sigmoid", color="#1f77b4", lw=2)
ax[0].plot(z_range.detach().numpy(), tanh.detach().numpy(), label="Tanh", color="#ff7f0e", lw=2)
ax[0].plot(z_range.detach().numpy(), relu.detach().numpy(), label="ReLU", color="#2ca02c", lw=2)
ax[0].set_title(r"Activation Functions $\phi(z)$")
ax[0].set_xlabel("Preactivation $z$")
ax[0].set_ylabel(r"Activation $\phi(z)$")
ax[0].grid(True, alpha=0.3)
ax[0].legend()

ax[1].plot(z_range.detach().numpy(), sig_grad.numpy(), label=r"Sigmoid $\sigma'(z) \leq 0.25$", color="#1f77b4", lw=2)
ax[1].plot(z_range.detach().numpy(), tanh_grad.numpy(), label=r"Tanh $\tanh'(z) \leq 1.0$", color="#ff7f0e", lw=2)
ax[1].plot(z_range.detach().numpy(), relu_grad.numpy(), label=r"ReLU $\phi'(z) \in \{0, 1\}$", color="#2ca02c", lw=2)
ax[1].set_title(r"Local Derivatives $\phi'(z)$")
ax[1].set_xlabel("Preactivation $z$")
ax[1].set_ylabel(r"$\phi'(z)$")
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "activation_derivatives.png", dpi=150)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'activation_derivatives.png'}")

## RUN 8 Failure mode: Deep Sigmoid vanishing gradient in a 10-layer network

**ASK BEFORE RUNNING:** Observe the gradient norm shrink from Layer 10 down to Layer 1. Why does an exponential decay occur?

In [ ]:
# When stacking 10 sigmoid layers with small weights, gradients shrink exponentially by (0.25)^L
layers = 10
dim = 64
batch = 128
X_deep = torch.randn(batch, dim)

def build_deep_net(act_cls, init_scale=1.0):
    net = nn.Sequential()
    for l in range(layers):
        linear = nn.Linear(dim, dim)
        nn.init.normal_(linear.weight, mean=0.0, std=init_scale)
        nn.init.zeros_(linear.bias)
        net.add_module(f"layer_{l}", linear)
        net.add_module(f"act_{l}", act_cls())
    return net

deep_sigmoid = build_deep_net(nn.Sigmoid, init_scale=1.0)
out_sig = deep_sigmoid(X_deep)
loss_sig = out_sig.sum()
loss_sig.backward()

grad_norms_sigmoid = []
for l in range(layers):
    w = deep_sigmoid[l*2].weight
    grad_norms_sigmoid.append(w.grad.norm().item())

print("Layer-by-layer weight gradient norms in 10-layer Sigmoid network:")
for l, norm in enumerate(grad_norms_sigmoid):
    print(f"  Layer {l+1:2d}: {norm:.4e}")
print(f"Ratio Layer 1 / Layer 10: {grad_norms_sigmoid[0] / grad_norms_sigmoid[-1]:.4e}")

## RUN 9 Symmetry collapse under zero initialization

**ASK BEFORE RUNNING:** If all weights and biases are initialized to 0, what happens to the gradients of different hidden units?

In [ ]:
# Prove that initializing weights to 0 prevents hidden units from differentiating
torch.manual_seed(42)
net_zeros = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 2)
)
nn.init.zeros_(net_zeros[0].weight)
nn.init.zeros_(net_zeros[0].bias)
nn.init.zeros_(net_zeros[2].weight)
nn.init.zeros_(net_zeros[2].bias)

X_dummy = torch.randn(8, 2)
y_dummy = torch.randint(0, 2, (8,))
criterion = nn.CrossEntropyLoss()

out_zeros = net_zeros(X_dummy)
loss_zeros = criterion(out_zeros, y_dummy)
loss_zeros.backward()

print("Layer 1 weight gradient under zero initialization:")
print(net_zeros[0].weight.grad)
# Notice all rows in Layer 1 weight gradient are identical because all hidden units received identical error signals!
row_diff = torch.norm(net_zeros[0].weight.grad[0] - net_zeros[0].weight.grad[1]).item()
print(f"Difference between hidden unit 0 and 1 gradients: {row_diff:.6f} (Symmetry unbroken!)")

# Break

20 minutes. Resume with signal propagation and variance analysis.

## RUN 10 Signal propagation across layers: Variance analysis

**ASK BEFORE RUNNING:** Derive why Var(y) = d_in * Var(w) * Var(x) and why ReLU requires an extra factor of 2 (Kaiming initialization).

In [ ]:
# For linear layer: Var(y) = d_in * Var(w) * Var(x)
# To preserve variance: Var(w) = 1 / d_in (LeCun) or 2 / (d_in + d_out) (Xavier/Glorot)
# For ReLU: half the units are killed, so Var(y) = (d_in / 2) * Var(w) * Var(x)
# Therefore He/Kaiming initialization requires: Var(w) = 2 / d_in -> std = sqrt(2 / d_in)
d_in = 256
d_out = 256
n_samples = 10000
x_init = torch.randn(n_samples, d_in)  # Var = 1.0

# 1. Too small (std = 0.01)
w_small = torch.randn(d_in, d_out) * 0.01
# 2. Too large (std = 1.0)
w_large = torch.randn(d_in, d_out) * 1.0
# 3. Xavier (std = sqrt(2 / (d_in + d_out)))
std_xavier = math.sqrt(2.0 / (d_in + d_out))
w_xavier = torch.randn(d_in, d_out) * std_xavier
# 4. Kaiming/He (std = sqrt(2 / d_in))
std_kaiming = math.sqrt(2.0 / d_in)
w_kaiming = torch.randn(d_in, d_out) * std_kaiming

print(f"Input variance: {torch.var(x_init).item():.4f}")
print(f"Linear output variance with small std (0.01): {torch.var(x_init @ w_small).item():.6f}")
print(f"Linear output variance with large std (1.00): {torch.var(x_init @ w_large).item():.2f}")
print(f"Linear output variance with Xavier std ({std_xavier:.4f}): {torch.var(x_init @ w_xavier).item():.4f}")
print(f"ReLU output variance with Xavier: {torch.var(torch.relu(x_init @ w_xavier)).item():.4f} (Variance halved!)")
print(f"ReLU output variance with Kaiming ({std_kaiming:.4f}): {torch.var(torch.relu(x_init @ w_kaiming)).item():.4f} (Preserved!)")

## RUN 11 Empirical distribution of activations across 8 layers: Naive vs Xavier vs Kaiming

**ASK BEFORE RUNNING:** Inspect the activation histograms across 8 layers. Which initialization maintains a non-degenerate variance?

In [ ]:
# Pass input through 8 layers with ReLU activation under 3 schemes
n_layers = 8
dim = 256
batch_sz = 1024
x_test = torch.randn(batch_sz, dim)

schemes = {
    'Naive Small (std=0.05)': lambda d_in, d_out: torch.randn(d_in, d_out) * 0.05,
    'Naive Large (std=0.35)': lambda d_in, d_out: torch.randn(d_in, d_out) * 0.35,
    'Xavier Normal': lambda d_in, d_out: torch.randn(d_in, d_out) * math.sqrt(2.0 / (d_in + d_out)),
    'Kaiming/He Normal': lambda d_in, d_out: torch.randn(d_in, d_out) * math.sqrt(2.0 / d_in)
}

hist_data = {k: [] for k in schemes}

for name, init_fn in schemes.items():
    h = x_test.clone()
    for l in range(n_layers):
        W = init_fn(dim, dim)
        z = h @ W
        h = torch.relu(z)
        if l in [0, 2, 4, 7]:
            hist_data[name].append((l+1, h.detach().flatten().numpy()))

fig, axes = plt.subplots(4, 4, figsize=(14, 10), sharex=True, sharey=True)
layer_indices = [1, 3, 5, 8]

for row_idx, (name, records) in enumerate(hist_data.items()):
    for col_idx, (layer_num, acts) in enumerate(records):
        ax = axes[row_idx, col_idx]
        # Filter non-zero activations for cleaner log/std visualization
        ax.hist(acts, bins=40, range=(0, 6), density=True, color='#2b5c8f', alpha=0.7)
        var_val = np.var(acts)
        ax.set_title(f"{name}\nL{layer_num} Var={var_val:.3f}", fontsize=8)
        if row_idx == 3:
            ax.set_xlabel("Activation value")
        if col_idx == 0:
            ax.set_ylabel("Density")
        ax.grid(True, alpha=0.2)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "layer_activation_distributions.png", dpi=150)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'layer_activation_distributions.png'}")

## RUN 12 Inverted Dropout mathematics and implementation

**ASK BEFORE RUNNING:** Why is inverted dropout scaled by 1/(1-p) during training? What is the expected value of an activated unit?

In [ ]:
# Inverted dropout:
# Training: mask ~ Bernoulli(1 - p), h_drop = (h * mask) / (1 - p)
# Inference: h_drop = h (identity)
p_drop = 0.4
keep_p = 1.0 - p_drop
torch.manual_seed(777)
h_sample = torch.ones(5, 8) * 2.0  # mean = 2.0

# Scratch inverted dropout
def scratch_dropout(h, p, training=True):
    if not training or p == 0.0:
        return h
    mask = (torch.rand_like(h) > p).float()
    return (h * mask) / (1.0 - p)

h_train_drop = scratch_dropout(h_sample, p_drop, training=True)
h_eval_drop = scratch_dropout(h_sample, p_drop, training=False)

print(f"Input mean: {h_sample.mean().item():.4f}")
print(f"Dropout during training (p={p_drop}):")
print(f"  Output sample row 0:\n  {h_train_drop[0]}")
print(f"  Training mean: {h_train_drop.mean().item():.4f} (Expectation preserved!)")
print(f"Dropout during eval (training=False):")
print(f"  Output mean: {h_eval_drop.mean().item():.4f} (Exact identity!)")

## RUN 13 PyTorch nn.Dropout train() vs eval() contract

**ASK BEFORE RUNNING:** What happens when model.eval() is called? How does nn.Dropout know whether to drop units or act as identity?

In [ ]:
torch_dropout = nn.Dropout(p=p_drop)
torch_dropout.train()
th_train = torch_dropout(h_sample)
torch_dropout.eval()
th_eval = torch_dropout(h_sample)

print(f"nn.Dropout in train() mode mean: {th_train.mean().item():.4f}")
print(f"nn.Dropout in eval() mode matches input exactly: {torch.equal(th_eval, h_sample)}")

## RUN 14 Synthetic diagnostic dataset: Multi-ring concentric dataset

**ASK BEFORE RUNNING:** Inspect the 3-ring concentric synthetic dataset. Is this linearly separable? What capacity is needed?

In [ ]:
# 3 concentric rings with label {0, 1, 2} and controlled Gaussian jitter
def generate_ring_dataset(n_per_ring=300, noise=0.08, seed=42):
    np.random.seed(seed)
    radii = [0.8, 1.8, 2.8]
    X_list, y_list = [], []
    for cls, r in enumerate(radii):
        angles = np.random.uniform(0, 2*np.pi, n_per_ring)
        radial_noise = np.random.normal(0, noise, n_per_ring)
        x1 = (r + radial_noise) * np.cos(angles)
        x2 = (r + radial_noise) * np.sin(angles)
        X_list.append(np.column_stack([x1, x2]))
        y_list.append(np.full(n_per_ring, cls))
    X = np.vstack(X_list).astype(np.float32)
    y = np.concatenate(y_list).astype(np.int64)
    # Shuffle
    idx = np.random.permutation(len(y))
    return torch.tensor(X[idx]), torch.tensor(y[idx])

X_all, y_all = generate_ring_dataset(n_per_ring=400, noise=0.09, seed=123)
N_tot = len(y_all)
n_train = int(0.6 * N_tot)
n_val = int(0.2 * N_tot)
n_test = N_tot - n_train - n_val

X_train, y_train = X_all[:n_train], y_all[:n_train]
X_val, y_val = X_all[n_train:n_train+n_val], y_all[n_train:n_train+n_val]
X_test, y_test = X_all[n_train+n_val:], y_all[n_train+n_val:]

print(f"Dataset generated:")
print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"  Class balance: {torch.bincount(y_train).tolist()} across 3 classes")

## RUN 15 Model definition: 4-layer MLP with modular initialization and dropout

**ASK BEFORE RUNNING:** Examine the modular DeepMLP architecture. How are custom weight initializations applied across all linear modules?

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, in_features=2, hidden_features=64, num_classes=3, 
                 p_dropout=0.0, init_mode='kaiming'):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, hidden_features)
        self.fc4 = nn.Linear(hidden_features, num_classes)
        self.dropout = nn.Dropout(p=p_dropout)
        self.relu = nn.ReLU()
        
        # Apply initialization
        self._init_weights(init_mode)
        
    def _init_weights(self, mode):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if mode == 'kaiming':
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                elif mode == 'xavier':
                    nn.init.xavier_normal_(m.weight)
                elif mode == 'naive_small':
                    nn.init.normal_(m.weight, mean=0.0, std=0.01)
                elif mode == 'naive_large':
                    nn.init.normal_(m.weight, mean=0.0, std=1.2)
                elif mode == 'zeros':
                    nn.init.zeros_(m.weight)
                nn.init.zeros_(m.bias)
                
    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        x = self.fc4(x)
        return x

test_m = DeepMLP(init_mode='kaiming')
print(f"DeepMLP instantiated. Total trainable parameters: {sum(p.numel() for p in test_m.parameters())}")

## RUN 16 Training engine with gradient norm tracking and evaluation

**ASK BEFORE RUNNING:** Trace the training loop: where is zero_grad() called, where is loss.backward() evaluated, and what gradients are logged?

In [ ]:
def train_and_evaluate(model, X_tr, y_tr, X_va, y_va, epochs=120, lr=0.02, batch_size=64):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    
    n_batches = math.ceil(len(X_tr) / batch_size)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'grad_norm_fc1': [], 'grad_norm_fc4': []}
    
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr))
        epoch_loss = 0.0
        correct = 0
        
        for b in range(n_batches):
            b_idx = perm[b*batch_size : (b+1)*batch_size]
            xb, yb = X_tr[b_idx], y_tr[b_idx]
            
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            
            # Record gradient norms of earliest and latest layer on first batch
            if b == 0:
                history['grad_norm_fc1'].append(model.fc1.weight.grad.norm().item())
                history['grad_norm_fc4'].append(model.fc4.weight.grad.norm().item())
                
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
            correct += (out.argmax(dim=1) == yb).sum().item()
            
        epoch_loss /= len(X_tr)
        epoch_acc = correct / len(X_tr)
        
        # Validation pass
        model.eval()
        with torch.no_grad():
            v_out = model(X_va)
            v_loss = criterion(v_out, y_va).item()
            v_acc = (v_out.argmax(dim=1) == y_va).float().mean().item()
            
        history['train_loss'].append(epoch_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(epoch_acc)
        history['val_acc'].append(v_acc)
        
    return history

print("Training engine compiled.")

## RUN 17 Comparison: Naive Small vs Naive Large vs Kaiming Initialization

**ASK BEFORE RUNNING:** Compare learning progress under small variance (0.01) vs large variance vs Kaiming. Why does small init get stuck at 33% (chance)?

In [ ]:
torch.manual_seed(42)
m_naive_small = DeepMLP(p_dropout=0.0, init_mode='naive_small')
torch.manual_seed(42)
m_naive_large = DeepMLP(p_dropout=0.0, init_mode='naive_large')
torch.manual_seed(42)
m_kaiming = DeepMLP(p_dropout=0.0, init_mode='kaiming')

hist_small = train_and_evaluate(m_naive_small, X_train, y_train, X_val, y_val, epochs=80, lr=0.03)
hist_large = train_and_evaluate(m_naive_large, X_train, y_train, X_val, y_val, epochs=80, lr=0.03)
hist_kaiming = train_and_evaluate(m_kaiming, X_train, y_train, X_val, y_val, epochs=80, lr=0.03)

print(f"Final Validation Accuracy after 80 epochs:")
print(f"  Naive Small:  {hist_small['val_acc'][-1]*100:.2f}%")
print(f"  Naive Large:  {hist_large['val_acc'][-1]*100:.2f}%")
print(f"  Kaiming (He): {hist_kaiming['val_acc'][-1]*100:.2f}%")

## RUN 18 Learning curve visualization of initialization dynamics

**ASK BEFORE RUNNING:** Examine the learning curves and first-layer gradient norms. Why did Naive Small init suffer from gradient starvation?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Loss
axes[0].plot(hist_small['train_loss'], label='Naive Small (0.01)', color='#d62728', ls='--')
axes[0].plot(hist_large['train_loss'], label='Naive Large (1.20)', color='#ff7f0e', ls='-.')
axes[0].plot(hist_kaiming['train_loss'], label='Kaiming Normal', color='#2ca02c', lw=2)
axes[0].set_title("Training Loss Dynamics")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(hist_small['val_acc'], label='Naive Small', color='#d62728', ls='--')
axes[1].plot(hist_large['val_acc'], label='Naive Large', color='#ff7f0e', ls='-.')
axes[1].plot(hist_kaiming['val_acc'], label='Kaiming Normal', color='#2ca02c', lw=2)
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gradient norms at Layer 1
axes[2].plot(hist_small['grad_norm_fc1'], label='Naive Small L1 Grad', color='#d62728', ls='--')
axes[2].plot(hist_large['grad_norm_fc1'], label='Naive Large L1 Grad', color='#ff7f0e', ls='-.')
axes[2].plot(hist_kaiming['grad_norm_fc1'], label='Kaiming L1 Grad', color='#2ca02c', lw=2)
axes[2].set_title("First Layer (fc1) Gradient Norm")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel(r"$\|\nabla_{W^{(1)}} L\|_2$")
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "initialization_comparison_curves.png", dpi=150)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'initialization_comparison_curves.png'}")

## RUN 19 Comparison: No-Dropout vs Regularized Inverted Dropout

**ASK BEFORE RUNNING:** Compare the training vs validation gap with and without dropout. How does dropout reduce generalization gap?

In [ ]:
torch.manual_seed(42)
m_no_drop = DeepMLP(p_dropout=0.0, init_mode='kaiming')
torch.manual_seed(42)
m_with_drop = DeepMLP(p_dropout=0.25, init_mode='kaiming')

hist_no_drop = train_and_evaluate(m_no_drop, X_train, y_train, X_val, y_val, epochs=120, lr=0.03)
hist_with_drop = train_and_evaluate(m_with_drop, X_train, y_train, X_val, y_val, epochs=120, lr=0.03)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(hist_no_drop['train_loss'], 'b--', label='Train (No Dropout)')
axes[0].plot(hist_no_drop['val_loss'], 'b-', label='Val (No Dropout)')
axes[0].plot(hist_with_drop['train_loss'], 'g--', label='Train (Dropout p=0.25)')
axes[0].plot(hist_with_drop['val_loss'], 'g-', label='Val (Dropout p=0.25)')
axes[0].set_title("Loss Curves: Overfitting vs Regularization")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(hist_no_drop['train_acc'], 'b--', label='Train Acc (No Drop)')
axes[1].plot(hist_no_drop['val_acc'], 'b-', label='Val Acc (No Drop)')
axes[1].plot(hist_with_drop['train_acc'], 'g--', label='Train Acc (Dropout)')
axes[1].plot(hist_with_drop['val_acc'], 'g-', label='Val Acc (Dropout)')
axes[1].set_title("Generalization Accuracy Gap")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "dropout_regularization_curves.png", dpi=150)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'dropout_regularization_curves.png'}")

## RUN 20 In-class live debugging exercises: 3 common training pathologies

**ASK BEFORE RUNNING:** Identify 3 bugs: missing zero_grad(), forgetting model.eval() with dropout, and graph leaks.

In [ ]:
print("--- RUN 20: Diagnostic Debugging Demos ---")

# Bug 1: Missing optimizer.zero_grad()
net_bug1 = nn.Linear(2, 2)
opt_bug1 = torch.optim.SGD(net_bug1.parameters(), lr=0.01)
inp_bug = torch.randn(4, 2)
tar_bug = torch.tensor([0, 1, 0, 1])
crit = nn.CrossEntropyLoss()

grad_accum_history = []
for step in range(5):
    # OMISSION: opt_bug1.zero_grad() is commented out!
    loss = crit(net_bug1(inp_bug), tar_bug)
    loss.backward()
    grad_accum_history.append(net_bug1.weight.grad.norm().item())
print(f"Bug 1 (Accumulating Gradients without zero_grad):")
print(f"  Gradient norm per step: {[round(v, 2) for v in grad_accum_history]} (Grows linearly!)")

# Bug 2: Missing model.eval() during inference with dropout
net_bug2 = DeepMLP(p_dropout=0.5, init_mode='kaiming')
net_bug2.train() # Left in train mode during test evaluation
test_x = X_test[:10]
pred_runs = [net_bug2(test_x).argmax(dim=1).tolist() for _ in range(3)]
print(f"Bug 2 (Evaluation left in train mode with dropout):")
for r, preds in enumerate(pred_runs):
    print(f"  Evaluation pass {r+1} predictions: {preds} (Nondeterministic!)")

net_bug2.eval() # Correct evaluation mode
eval_preds = [net_bug2(test_x).argmax(dim=1).tolist() for _ in range(3)]
print(f"Correct eval mode:")
print(f"  Evaluation passes identical: {eval_preds[0] == eval_preds[1] == eval_preds[2]}")

## RUN 21 Final test report on frozen test partition

**ASK BEFORE RUNNING:** State everything frozen before evaluating the held-out test set. How do the initializations compare on test loss?

In [ ]:
print("--- RUN 21: Final Frozen Test Set Evaluation ---")
criterion = nn.CrossEntropyLoss()

def evaluate_test(model, name):
    model.eval()
    with torch.no_grad():
        out = model(X_test)
        loss = criterion(out, y_test).item()
        acc = (out.argmax(dim=1) == y_test).float().mean().item()
    return {'model': name, 'test_loss': loss, 'test_acc': acc}

results = [
    evaluate_test(m_naive_small, "MLP (Naive Small Init 0.01)"),
    evaluate_test(m_naive_large, "MLP (Naive Large Init 1.20)"),
    evaluate_test(m_kaiming, "MLP (Kaiming Normal Init, No Dropout)"),
    evaluate_test(m_with_drop, "MLP (Kaiming Normal Init + Dropout 0.25)")
]

for res in results:
    print(f"  {res['model']:<45}: Loss = {res['test_loss']:.4f}, Acc = {res['test_acc']*100:.2f}%")

with open(OUTPUT_DIR / 'experiment_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved {OUTPUT_DIR / 'experiment_results.json'}")

## RUN 22 RESERVE: Gradient flow visualization across all layers

**ASK BEFORE RUNNING:** RESERVE: Inspect gradient norms across layers 1 to 4. Does Kaiming maintain healthy gradient flow throughout?

In [ ]:
# Diagnostic bar plot of gradient norms across layers 1 to 4
fig, ax = plt.subplots(figsize=(7, 4))
layers_list = ['fc1', 'fc2', 'fc3', 'fc4']
grad_norms_kaiming = [
    m_kaiming.fc1.weight.grad.norm().item(),
    m_kaiming.fc2.weight.grad.norm().item(),
    m_kaiming.fc3.weight.grad.norm().item(),
    m_kaiming.fc4.weight.grad.norm().item()
]
ax.bar(layers_list, grad_norms_kaiming, color='#2b5c8f', width=0.5)
ax.set_title(r"Gradient Norm Across Layers: Stable Backprop ($\|\nabla_{W^{(l)}} L\|_2$)")
ax.set_xlabel("Layer")
ax.set_ylabel("Frobenius Norm of Weight Gradient")
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "layer_gradient_flow.png", dpi=150)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'layer_gradient_flow.png'}")